#### 3. scRNA-seq processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, shutil, importlib, glob
import celloracle as co
import gimmemotifs
import faulthandler
faulthandler.enable()
import gc
gc.collect()
import scanpy as sc
import anndata as ad
import muon as mu

from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
from tqdm.notebook import tqdm
%matplotlib inline

In [ ]:
prefix = "/home/fuyq/GRN/single_cell/celloracle/pbmc10k/result/cellorcale/seurat_output/"
counts = sc.read_mtx(prefix + "rna_counts.mtx").T
barcodes = pd.read_csv(prefix + "rna_barcodes.tsv", header=None)[0].values
genes = pd.read_csv(prefix + "rna_genes.tsv", header=None)[0].values

In [ ]:
adata = sc.AnnData(X=counts)
adata.obs_names = barcodes
adata.var_names = genes
metadata = pd.read_csv(prefix + "metadata.csv", index_col=0)
metadata = metadata.loc[adata.obs_names]
adata.obs = metadata

In [ ]:
original_counts = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
# Combine TF + HVG genes
rna=adata
highly_variable_mask = rna.var['highly_variable'].values
print("Number of highly variable genes:", rna.var['highly_variable'].sum())
with open("/home/fuyq/GRN/GTEx/tf_hg_new.txt") as f:
    tf_genes = [line.strip() for line in f if line.strip() != '']
hvg_genes = rna.var_names[rna.var["highly_variable"]].tolist()
combined_genes = list(set(tf_genes).union(set(hvg_genes)))
final_genes = [g for g in combined_genes if g in rna.var_names]
rna = rna[:, final_genes].copy()
gene_indices = [rna.var_names.get_loc(g) for g in final_genes if g in rna.var_names]
rna.layers["raw_count"] = original_counts[:, gene_indices]

In [ ]:
sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, svd_solver='arpack')
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=20)
sc.tl.leiden(rna, resolution=.5)
sc.tl.umap(rna, spread=1., min_dist=.5, random_state=11)
sc.pl.umap(rna, color="leiden", legend_loc="on data")
sc.tl.paga(rna, groups='leiden')
plt.rcParams["figure.figsize"] = [6, 4.5]
sc.pl.paga(rna)
sc.tl.draw_graph(rna, init_pos='paga', random_state=123)
sc.pl.draw_graph(rna, color='leiden', legend_loc='on data')
sc.pl.umap(rna, color="celltype", legend_loc="on data", legend_fontsize=6)
